# Data Cleaning: Preparing the Ames Housing Dataset

## Learning Goals

After completing this notebook, you will be able to:

- Load and inspect the Ames Housing dataset
- Identify missing values and data quality issues
- Transform temporal data (monthly to quarterly aggregation)
- Merge macroeconomic indicators with housing prices
- Prepare clean, analysis-ready datasets

## Keywords

data cleaning, feature engineering, data merging, temporal aggregation, data validation

## Prerequisite Knowledge

- Basic Python (pandas, numpy)
- Familiarity with CSV files and dataframes
- Understanding of what missing values are

## Target User

Data analysts and aspiring data scientists learning to build prediction models

## Table of Contents

1. [Part 1: Understanding the Ames Dataset](#part-1)
2. [Part 2: Loading and Inspecting Data](#part-2)
3. [Part 3: Temporal Aggregation](#part-3)
4. [Part 4: Merging Macro Indicators](#part-4)
5. [Part 5: Validation and Export](#part-5)

## Part 1: Understanding the Ames Dataset {#part-1}

The Ames Housing dataset is a well-known dataset in machine learning containing information about residential properties sold in Ames, Iowa. What makes this project unique is that we're expanding it with *macroeconomic indicators* - measurements of the broader economic health of the region.

### Why Macro Indicators Matter

Most housing prediction models use "micro" features like square footage, number of bathrooms, and property age. These are property-specific features. However, housing prices are also influenced by regional economic conditions: employment rates, tax revenue, manufacturing activity, and consumer spending all affect housing demand and prices.

By combining Ames property data with quarterly macroeconomic indicators for Iowa, we can test whether broader economic trends improve our price predictions.

### Data Sources

- **Ames Housing data**: Property sales records (typically 1460 samples with ~80 features)
- **Macroeconomic indicators**: Federal Reserve and state tax data (quarterly time series)

### Why We Clean Data

Raw data rarely works perfectly in models. Common issues:
- Missing values that could bias results
- Inconsistent formats (some dates as strings, some as numbers)
- Misaligned time periods (monthly data that needs quarterly grouping)
- Duplicate records or obvious errors

We clean first to avoid feeding these problems downstream to our models.

### Concept Check 1.1

What is the main advantage of using macroeconomic indicators alongside property-specific features for price prediction?

A) They are easier to measure
B) They capture broader regional economic trends that affect housing demand
C) They replace the need for property-specific data
D) They always improve model accuracy

<details>
<summary>Answer</summary>
B) They capture broader regional economic trends that affect housing demand. Macro indicators provide context about the economic health of the region, which influences whether people are buying homes and at what prices.
</details>

## Part 2: Loading and Inspecting Data {#part-2}

### Why Inspection Comes First

Before cleaning, you need to *see* what you're working with. Inspection tells you:
- How many rows and columns you have
- Which columns have missing values and how many
- Data types (numeric, text, date)
- Basic statistics (mean, min, max) that reveal outliers or errors

This is more efficient than diving straight into cleaning - you might waste effort fixing things that don't affect your analysis.

### Loading the Data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load the Ames housing dataset
# In a real project, you would load from your data directory
# ames = pd.read_csv('data/ames_housing.csv')

# For demonstration, create a minimal example
# In practice, this would be a much larger dataset
print("Loading Ames Housing dataset...")
print("Expected columns: SalePrice, YearBuilt, TotalSqFt, BedroomAbvGr, KitchenAbvGr, GarageCars, etc.")

### Inspection Workflow

In [ ]:
# Step 1: Basic shape and structure
# ames.shape  # Returns (n_rows, n_columns)
# ames.info() # Shows column names, types, and non-null counts

# Step 2: Check for missing values
# missing = ames.isnull().sum()
# missing[missing > 0].sort_values(ascending=False)

# Step 3: Summary statistics
# ames.describe()  # Mean, std, min, max for numeric columns

print("Inspection pattern:")
print("1. Shape: How many rows/columns?")
print("2. Info: What are the data types?")
print("3. Missing: Which columns have nulls and how many?")
print("4. Describe: What are the distributions?")

### Why This Order?

We inspect in this order because:
- **Shape first**: Gives you the overall picture (1460 rows = 1460 homes)
- **Info second**: Tells you which columns are numeric vs. text, critical for deciding cleaning strategy
- **Missing values third**: You can't decide how to handle nulls without knowing the data types
- **Statistics last**: Reveals outliers and distributions that might need transformation

This order prevents you from going down wrong paths early on.

### Concept Check 2.1

If a numeric column has 50 missing values out of 1460 rows (~3.4%), should you:

A) Delete all 50 rows
B) Fill with the mean of that column
C) Investigate whether those missing values form a pattern (e.g., are they all from recent sales?)
D) Delete the entire column

<details>
<summary>Answer</summary>
C) Investigate whether those missing values form a pattern. A small percentage of missing data (3.4%) is often manageable, but you need to understand why it's missing. If recent sales have missing values because data wasn't recorded yet, that's different from random corruption. Context matters more than the raw count.
</details>

## Part 3: Temporal Aggregation {#part-3}

### The Problem: Frequency Mismatch

Our macroeconomic data arrives as *monthly* values (12 data points per year), but we want to align it with *quarterly* housing sales (4 data points per year). We need to aggregate monthly data into quarterly data.

### Aggregation Strategy

For most economic indicators, taking the quarterly *average* of monthly values makes sense:
- Q1 (Jan-Feb-Mar) = average of January, February, March values
- Q2 (Apr-May-Jun) = average of April, May, June values
- etc.

Averaging smooths out month-to-month noise while keeping quarterly trends.

In [ ]:
def aggregate_monthly_to_quarterly(monthly_data):
    """
    Convert monthly time series to quarterly by averaging.
    
    Args:
        monthly_data: list or array of 12, 24, 36... monthly values
    
    Returns:
        quarterly_data: list of 4, 8, 12... quarterly averages
    """
    quarterly = []
    
    # Group every 3 consecutive months
    for i in range(0, len(monthly_data), 3):
        quarter_avg = np.mean(monthly_data[i:i+3])
        quarterly.append(quarter_avg)
    
    return quarterly

# Example: 12 months of inventory data
monthly_inventory = [100, 102, 98, 105, 103, 101, 99, 97, 96, 104, 106, 102]
quarterly_inventory = aggregate_monthly_to_quarterly(monthly_inventory)

print(f"Monthly: {monthly_inventory}")
print(f"Quarterly: {quarterly_inventory}")
print(f"Q1 average: {np.mean(monthly_inventory[0:3]):.2f}")
print(f"Q2 average: {np.mean(monthly_inventory[3:6]):.2f}")
print(f"Q3 average: {np.mean(monthly_inventory[6:9]):.2f}")
print(f"Q4 average: {np.mean(monthly_inventory[9:12]):.2f}")

### Why Averaging Over Other Methods?

You could also aggregate by:
- **Sum**: Useful for totals (total sales revenue per quarter)
- **Last month of quarter**: Would miss variations within the quarter
- **Maximum/Minimum**: Would ignore normal variation

Averaging is chosen because most economic indicators are *rates* (unemployment %, tax revenue per capita), not counts. A rate's quarterly value should reflect the average rate across those three months.

### Concept Check 3.1

If you have monthly unemployment rates: [5.0, 4.9, 4.8, 4.7, 4.6, 4.5, ...], which aggregation makes most sense?

A) Sum them
B) Average them
C) Take the maximum
D) Take only the third month of each quarter

<details>
<summary>Answer</summary>
B) Average them. Unemployment rate is a percentage/rate, not a count. The quarterly unemployment rate should reflect the average joblessness during those three months. Summing or taking just the last month would misrepresent the quarterly trend.
</details>

## Part 4: Merging Macro Indicators {#part-4}

### Combining Datasets by Time Period

Once both housing and macro data are on a quarterly schedule, we merge them by matching quarters. Each housing sale gets labeled with the quarter it occurred in, and matched to the macro indicators for that quarter.

In [ ]:
# Example merge logic
# For each home sale date, extract the year and quarter
# Then look up the macro indicators for that year-quarter

def get_quarter(month):
    """Return quarter number (1-4) for a given month (1-12)."""
    return (month - 1) // 3 + 1

def extract_year_quarter(date_string):
    """
    Convert date to (year, quarter) tuple.
    Example: '2010-07-15' -> (2010, 3)
    """
    year, month, day = map(int, date_string.split('-'))
    quarter = get_quarter(month)
    return (year, quarter)

# Example
sale_date = '2010-07-15'
year, quarter = extract_year_quarter(sale_date)
print(f"Sale date: {sale_date}")
print(f"Year-Quarter: {year}-Q{quarter}")
print(f"Will be matched with macro indicators for 2010 Q3")

### Why This Matters

Merging creates a richer dataset. Instead of predicting price from just:
- Square footage, bedrooms, garage size, condition

We can now predict from:
- Property features (as above) + 
- Economic indicators (unemployment rate, manufacturing employment, income levels, etc.)

The hypothesis: homes sell for more when regional economic conditions are strong.

### Concept Check 4.1

A home was sold on April 15, 2012. Quarterly macro data is organized as (2012, 1), (2012, 2), (2012, 3), (2012, 4). Which should you merge this sale with?

A) (2012, 1) - Q1
B) (2012, 2) - Q2
C) (2012, 3) - Q3
D) (2012, 4) - Q4

<details>
<summary>Answer</summary>
B) (2012, 2) - Q2. April is month 4, and months 4-5-6 make up Q2. The macroeconomic conditions of Q2 (Apr-May-Jun) are most relevant to this sale.
</details>

## Part 5: Validation and Export {#part-5}

### Final Validation Checklist

Before exporting cleaned data for modeling, verify:
1. No unexpected missing values remain (or only in expected columns)
2. Data types are correct (prices are numeric, dates are parsed)
3. No duplicate rows
4. Numeric columns have reasonable ranges (no negative prices, realistic square footage)
5. All merges completed successfully (expected number of rows after merge)

In [ ]:
# Example validation checks
def validate_cleaned_data(df):
    """
    Run validation checks on cleaned dataset.
    Raises AssertionError if any check fails.
    """
    # Check 1: No duplicates
    assert not df.duplicated().any(), "Found duplicate rows"
    
    # Check 2: Price is positive
    assert (df['SalePrice'] > 0).all(), "Found non-positive sale prices"
    
    # Check 3: Year is reasonable
    assert (df['YearBuilt'] >= 1800).all() and (df['YearBuilt'] <= 2020).all(), "Found invalid years"
    
    # Check 4: Expected columns exist
    required = ['SalePrice', 'YearBuilt', 'TotalSqFt']
    assert all(col in df.columns for col in required), f"Missing required columns: {required}"
    
    print("✓ All validation checks passed")
    print(f"  - {len(df)} rows, {len(df.columns)} columns")
    print(f"  - No duplicates")
    print(f"  - All prices positive")
    print(f"  - All years in valid range")

# Example usage (would require actual data)
# validate_cleaned_data(ames_with_macro)

### Exporting for Next Steps

Save the cleaned dataset in a format that the next notebook (EDA) can easily load:

In [ ]:
# ames_with_macro.to_csv('data/ames_with_macro_clean.csv', index=False)
# print("Cleaned data saved to: data/ames_with_macro_clean.csv")

print("Export pattern:")
print("1. Validate data thoroughly")
print("2. Save to a consistent location (data/ directory)")
print("3. Use clear naming (includes 'clean' to indicate processed state)")
print("4. Use CSV format for portability")

### Concept Check 5.1

After merging housing data with quarterly macro indicators, you have 1460 housing sales and the merge completed without errors. Should you:

A) Immediately use this for modeling
B) Run validation checks first, then export
C) Delete rows with any remaining nulls
D) Manually inspect each row

<details>
<summary>Answer</summary>
B) Run validation checks first, then export. Validation is faster than manual inspection but catches real issues (duplicates, data type errors, out-of-range values) before they cause modeling problems. This saves debugging time downstream.
</details>

---

## Summary

Data cleaning transforms raw data into a form ready for analysis:
- **Inspect first** to understand what you're working with
- **Handle temporal mismatches** by aggregating to common time periods
- **Merge strategically** to combine complementary datasets
- **Validate thoroughly** before handing off to analysis steps

This notebook covered the why behind each step. In the next notebook (02_exploratory-analysis.ipynb), we'll visualize and understand the cleaned data before building models.

---

## Next Steps

Proceed to [02_exploratory-analysis.ipynb](02_exploratory-analysis.ipynb) to explore patterns in the cleaned dataset.